# Introduction to NLP Fundamentals


In [1]:
!wget https://raw.githubusercontent.com/umerkang66/ai-ml-dl/refs/heads/master/tensorflow-bootcamp/03-computer-vision-tf/helper_functions.py

--2026-08-22 13:41:39--  https://raw.githubusercontent.com/umerkang66/ai-ml-dl/refs/heads/master/tensorflow-bootcamp/03-computer-vision-tf/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10836 (11K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.58K  --.-KB/s    in 0s      

2026-08-22 13:41:40 (85.6 MB/s) - ‘helper_functions.py’ saved [10836/10836]



In [2]:
from helper_functions import unzip_data, plot_loss_curves, compare_historys

## Kaggle Intro to NLP Dataset


In [3]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip

--2026-08-22 13:41:47--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.130.207, 74.125.68.207, 172.253.118.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.130.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K   681KB/s    in 0.9s    

2026-08-22 13:41:48 (681 KB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [4]:
unzip_data("nlp_getting_started.zip")

## Read the Data


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv("train.csv")[["text", "target"]]

train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

test_df = pd.read_csv("test.csv")[["text"]]

In [6]:
train_df.head()

,text,target
4620,'McFadden Reportedly to Test Hamstring Thursda...,0
2858,w--=-=-=-[ NEMA warns Nigerians to prepare for...,1
3098,When I was cooking earlier I got electrocuted ...,0
3751,I'm On Fire. http://t.co/WATsmxYTVa,0
5285,More than 40 families affected by the fatal ou...,1


## Shuffle the training data


In [7]:
train_df_shuffled = train_df.sample(frac=1, random_state=42)

In [8]:
train_df_shuffled.head()

,text,target
3347,78 passengers evacuated safely after Green Lin...,1
6177,@KIRO7Seattle Just saw a Bomb Squad car headin...,1
6970,@Kamunt Holy crap it's been forever since I sa...,0
1497,Learning from the Legacy of a Catastrophic Eru...,1
3453,luke + microphone = exploded ovaries,0


In [9]:
val_df.head()

,text,target
2644,So you have a new weapon that can cause un-ima...,1
2227,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,Aftershock back to school kick off was great. ...,0
6845,in response to trauma Children of Addicts deve...,0


In [10]:
test_df.head()

,text
0,Just happened a terrible car crash
1,"Heard about #earthquake is different cities, s..."
2,"there is a forest fire at spot pond, geese are..."
3,Apocalypse lighting. #Spokane #wildfires
4,Typhoon Soudelor kills 28 in China and Taiwan


In [11]:
train_df.target.value_counts()

,count
target,
0,3916
1,2935


In [12]:
len(train_df), len(test_df)

(6851, 3263)

## Let's visualize some random samples


In [13]:
import random

random_index = random.randint(0, len(train_df) - 1)

for _, row in train_df.iloc[random_index : random_index + 5].iterrows():
    text, target = row

    print(
        f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)"
    )
    print(f"Text: {text}\n")

Target: 0 (not real disaster)
Text: Army names 10th Mountain units for Iraq Afghanistan deployments (Deeds) http://t.co/N6ZfLXIGvr

Target: 1 (real disaster)
Text: I-10 EB at MS line update: Offloading the hazardous material is going much slower than expected. Road could stay closed until tomorrow AM.

Target: 0 (not real disaster)
Text: WWI WWII JAPANESE ARMY NAVY MILITARY JAPAN LEATHER WATCH WAR MIDO WW1 2 - Full read by eBay http://t.co/obfD7e4QcP http://t.co/yAZjE5OwVk

Target: 0 (not real disaster)
Text: Pandemonium In Aba As Woman Delivers Baby Without Face (Photos) - http://t.co/dI5aRr6HQ6

Target: 0 (not real disaster)
Text: Best windows torrent client? was recommended Deluge but it looks like it was written 10 years ago with java swing and 'uses' worse



## Converting text into numbers


In [14]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [15]:
average_max_length = round(
    sum([len(i.split()) for i in train_df["text"]]) / len(train_df)
)

In [16]:
# we have already defaulted this, but here we are setting again
max_length = average_max_length  # how many words from a tweet our model will see


text_vectorizer = TextVectorization(
    max_tokens=10000,  # None = no limit on the number of tokens vocab can have
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    ngrams=None,  # treat each word as a token
    output_mode="int",  # convert tokens into integers
    output_sequence_length=max_length,  # pad all outputs to be max_length tokens long
    pad_to_max_tokens=True,
)

In [17]:
# fit the text vectorizer to the training text

text_vectorizer.adapt(train_df["text"])

In [18]:
# create a sample text and pass it through our text vectorizer instance

sample_text = "There's a flood in my street!"

text_vectorizer([sample_text])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[282,   3, 206,   4,  13, 674,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [19]:
# choose a random text from the training dataset and pass it through the text vectorizer

import random

random_sentence = random.choice(train_df["text"])

print(f"Original text:\n{random_sentence} \n")
print(f"Vectorized version:\n{text_vectorizer([random_sentence])}")

Original text:
The Lightning out here is something serious! 

Vectorized version:
[[   2  313   36  127    9  503 1103    0    0    0    0    0    0    0
     0]]


In [20]:
# get the unique words in the vocabulary of our text vectorizer instance

words_in_vocab = text_vectorizer.get_vocabulary()

print("Top 5 words in vocab: ", words_in_vocab[:5])
print("Bottom 5 words in vocab: ", words_in_vocab[-5:])
print("Number of words in vocab: ", len(words_in_vocab))

Top 5 words in vocab:  ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab:  [np.str_('pakthey'), np.str_('pakistan\x89Ûªs'), np.str_('pakistans'), np.str_('pajamas'), np.str_('paints')]
Number of words in vocab:  10000


In [21]:
from tensorflow.keras.layers import Embedding

embedding = Embedding(
    input_dim=len(
        words_in_vocab
    ),  # total vocabulary size (i.e. number of unique tokens in the text)
    output_dim=128,  # set size of embedding vector
    input_length=max_length,  # how long is each input
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
random_sentence = random.choice(train_df["text"])


print(f"Original text:\n{random_sentence} \n")
print(f"Vectorized version:\n{text_vectorizer([random_sentence])}")

sample_embed = embedding(text_vectorizer([random_sentence]))


print(f"Embedded version:\n{sample_embed}")
print(f"Embedded version shape:\n{sample_embed.shape}")

Original text:
@devon_breneman hopefully it doesn't electrocute your heated blanket lmao 

Vectorized version:
[[   1 2956   15  605  603   35    1 4085 1002    0    0    0    0    0
     0]]
Embedded version:
[[[-3.2098949e-02 -1.8447362e-02  3.3189584e-02 ... -4.9450517e-02
    3.1672418e-05 -4.8248328e-02]
  [-4.0281307e-02  4.1993413e-02 -4.9074795e-02 ...  1.6656939e-02
   -3.0394185e-02  1.8466029e-02]
  [-4.3552089e-02 -3.3349946e-02  2.5550012e-02 ...  4.7012899e-02
   -1.7349876e-02 -8.6427331e-03]
  ...
  [ 2.2278797e-02 -6.7488104e-04 -3.0017912e-02 ...  2.7849782e-02
    3.8377229e-02 -2.4253523e-02]
  [ 2.2278797e-02 -6.7488104e-04 -3.0017912e-02 ...  2.7849782e-02
    3.8377229e-02 -2.4253523e-02]
  [ 2.2278797e-02 -6.7488104e-04 -3.0017912e-02 ...  2.7849782e-02
    3.8377229e-02 -2.4253523e-02]]]
Embedded version shape:
(1, 15, 128)


In [23]:
sample_embed[0][0], sample_embed[0][0].shape

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([-3.20989490e-02, -1.84473619e-02,  3.31895836e-02,  1.56510361e-02,
        -1.13020539e-02,  7.06434250e-03, -1.68571100e-02,  8.97980854e-03,
        -3.49194407e-02,  3.47630866e-02,  3.20938975e-03, -1.74581185e-02,
        -3.62703577e-02,  1.31706037e-02,  3.91789339e-02, -6.30297512e-03,
        -4.96545322e-02,  1.26077794e-02,  2.33575143e-02,  2.83195414e-02,
         1.71258561e-02,  1.81880482e-02, -1.35891214e-02,  4.95975502e-02,
         3.67788114e-02,  3.04655097e-02, -4.60191742e-02, -1.01592764e-02,
        -4.76769209e-02, -3.98068205e-02,  9.53450799e-05, -1.87624935e-02,
        -3.31163034e-02, -4.99570630e-02, -1.77775733e-02,  1.25899166e-03,
        -1.04449615e-02,  3.47608365e-02,  1.55499019e-02, -9.51730087e-03,
        -2.24174187e-03,  1.32600181e-02, -1.29488930e-02, -3.74928229e-02,
         2.74231099e-02, -3.76163125e-02, -4.76456285e-02, -1.95760261e-02,
        -2.97065265e-02, -2.49691606e-0

## Experiment Models

- Model 1: Feed Forward Neural Network (dense model)
- Model 2: LSTM model (RNN)
- Model 3: GRU model (RNN)
- Model 4: Bidirectional-LSTM (RNN)
- Model 5: 1D Convolutional Neural Network (CNN)
- Model 6: Transfer Learning for NLP


## BaseLine Naive Bayes
